# Telco Customer Churn Prediction

## 1. Problem Definition & Dataset Selection

**Objective:** To analyze the factors influencing customer churn and to build a predictive model that can identify customers who are likely to churn. This is a classification problem.
**Dataset:** `WA_Fn-UseC_-Telco-Customer-Churn.csv`. This dataset contains customer information and their churn status.

## 2. Data Cleaning & Preparation

In [7]:
# Import necessary libraries for data analysis and visualization
# pandas is used for data manipulation and analysis. 
# numpy is used for numerical operations. 
# matplotlib.pyplot and seaborn are used for data visualization.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set the style for our plots to make them look nice
sns.set(style="whitegrid")

In [8]:
# Load the dataset from the CSV file into a pandas DataFrame
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Let's take a look at the first few rows of our data to see what it looks like
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### 2.1 Handle Missing Values

In [ ]:
# We need to check if there are any missing values in our dataset.
# The 'TotalCharges' column has missing values. Let's find out how many.
# We convert the 'TotalCharges' column to numeric, coercing errors to NaN (Not a Number)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(f'Number of missing values in TotalCharges: {df["TotalCharges"]}.isnull().sum()')

# Since the number of missing values is small compared to the total number of rows, we can safely drop them.
df.dropna(inplace=True)

# Let's check again to make sure the missing values are gone.
print(f"Number of missing values in TotalCharges after cleaning: {df['TotalCharges'].isnull().sum()}")

SyntaxError: unterminated f-string literal (detected at line 5) (3592988145.py, line 5)

### 2.2 Remove Duplicates

In [ ]:
# Now, let's check for any duplicate rows in our data.
# Duplicate rows can skew our analysis, so it is important to remove them.
print(f'Number of duplicate rows: {df.duplicated()}.sum()')

# If there were duplicates, we would remove them using df.drop_duplicates(inplace=True)

### 2.3 Handle Outliers

In [ ]:
# We'll look for outliers in the numerical columns: 'tenure', 'MonthlyCharges', and 'TotalCharges'.
# A good way to spot outliers is to use boxplots.
# Outliers can affect the performance of our model, so it is important to identify them.
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
sns.boxplot(x=df['tenure'])
plt.title('Boxplot of Tenure')

plt.subplot(1, 3, 2)
sns.boxplot(x=df['MonthlyCharges'])
plt.title('Boxplot of Monthly Charges')

plt.subplot(1, 3, 3)
sns.boxplot(x=df['TotalCharges'])
plt.title('Boxplot of Total Charges')

plt.show()

The boxplots show that there are no significant outliers that need to be handled. The data points are within the whiskers of the boxplot, which indicates a normal distribution.

### 2.4 Encode Categorical Data

In [ ]:
# Our model can only understand numbers, so we need to convert our categorical columns into numerical ones.
# We'll use one-hot encoding for this. One-hot encoding creates new columns for each category and assigns a 1 or 0 to indicate the presence of the category.

# Create a copy of the dataframe to work with
df_encoded = df.copy()

# Encode the 'Churn' column to 1 for 'Yes' and 0 for 'No'
df_encoded['Churn'] = df_encoded['Churn'].apply(lambda x: 1 if x == 'Yes' else 0)

# Identify categorical columns for one-hot encoding, excluding 'customerID'
categorical_cols = df_encoded.select_dtypes(include=['object']).columns.tolist()
categorical_cols.remove('customerID')

# Perform one-hot encoding
df_encoded = pd.get_dummies(df_encoded, columns=categorical_cols, drop_first=True)

# Display the first few rows of the encoded dataframe
df_encoded.head()

## 3. Exploratory Data Analysis (EDA)

### 3.1 Univariate Analysis

In [ ]:
# Let's look at the distribution of our target variable, 'Churn'.
# This will help us understand the proportion of customers who churned.
plt.figure(figsize=(6, 4))
sns.countplot(x='Churn', data=df)
plt.title('Distribution of Customer Churn')
plt.show()

The distribution of churn shows that we have an imbalanced dataset, with more customers not churning than churning. This is important to keep in mind when building our models, as it can lead to a model that is biased towards the majority class. We may need to use techniques like oversampling or undersampling to address this imbalance.

In [ ]:
# Now let's look at the distributions of our numerical features.
# This will help us understand the characteristics of our numerical data.
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
sns.histplot(df['tenure'], kde=True)
plt.title('Distribution of Tenure')

plt.subplot(1, 3, 2)
sns.histplot(df['MonthlyCharges'], kde=True)
plt.title('Distribution of Monthly Charges')

plt.subplot(1, 3, 3)
sns.histplot(df['TotalCharges'], kde=True)
plt.title('Distribution of Total Charges')

plt.show()

The histograms show:
- **Tenure:** There is a bimodal distribution for tenure, with a large group of new customers (low tenure) and another large group of long-term customers (high tenure). This suggests that customers are either new to the service or have been with the company for a long time.
- **Monthly Charges:** The distribution of monthly charges is skewed to the left, with most customers having lower monthly charges. This could indicate that most customers are on basic plans.
- **Total Charges:** The distribution of total charges is skewed to the right, which is expected as long-term customers will have higher total charges. This is consistent with the tenure distribution.

### 3.2 Bivariate/Multivariate Analysis

In [ ]:
# Let's see how the numerical features relate to churn.
# This will help us understand if there is a relationship between these features and customer churn.
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
sns.boxplot(x='Churn', y='tenure', data=df)
plt.title('Tenure vs. Churn')

plt.subplot(1, 3, 2)
sns.boxplot(x='Churn', y='MonthlyCharges', data=df)
plt.title('Monthly Charges vs. Churn')

plt.subplot(1, 3, 3)
sns.boxplot(x='Churn', y='TotalCharges', data=df)
plt.title('Total Charges vs. Churn')

plt.show()

The boxplots suggest:
- **Tenure vs. Churn:** Customers who churn tend to have a lower tenure. This is a significant finding, as it suggests that new customers are more likely to churn. The company should focus on retaining new customers.
- **Monthly Charges vs. Churn:** Customers who churn tend to have higher monthly charges. This could be because they are not satisfied with the value they are getting for the price they are paying.
- **Total Charges vs. Churn:** Customers who churn tend to have lower total charges. This is likely because they have been with the company for a shorter period of time, as we saw in the tenure vs. churn plot.

In [ ]:
# Let's look at the correlation between our numerical features.
# The correlation matrix will give us a quantitative measure of the linear relationship between the variables.
correlation_matrix = df_encoded[['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()

The correlation matrix shows:
- **Tenure and Churn:** There is a negative correlation of -0.35 between tenure and churn, which supports our earlier observation that customers with lower tenure are more likely to churn.
- **MonthlyCharges and Churn:** There is a positive correlation of 0.19 between monthly charges and churn. This indicates that customers with higher monthly charges are slightly more likely to churn.
- **TotalCharges and Churn:** There is a negative correlation of -0.20 between total charges and churn. This is also consistent with our earlier findings.
- **Tenure and TotalCharges:** There is a strong positive correlation of 0.83 between tenure and total charges, which is expected. Customers who have been with the company longer will have higher total charges.